In [1]:
import pandas as pd
import numpy as np

In [2]:
df_1 = pd.read_csv('../transfermarkt_players.csv')

df_2 = pd.read_csv('../transfermarkt_player_valuations.csv')

In [3]:
df_1.head(10)

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,foot,height_in_cm,contract_expiration_date,agent_name,image_url,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,right,184.0,NaN,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,left,190.0,NaN,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,NaN,NaN,NaN,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,NaN,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,right,194.0,NaN,IFM,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0
5,109,NaN,Dedê,Dedê,2013,825,dede,Brazil,Belo Horizonte,Brazil,...,NaN,NaN,NaN,Football Concept,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/dede/profil/sp...,TR1,Eskisehirspor,400000.0,9500000.0
6,123,Christoph,Metzelder,Christoph Metzelder,2012,33,christoph-metzelder,Germany,Haltern,Germany,...,NaN,NaN,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/christoph-metz...,L1,FC Schalke 04,1500000.0,9500000.0
7,132,Tomas,Rosicky,Tomas Rosicky,2015,11,tomas-rosicky,CSSR,Praha,Czech Republic,...,both,179.0,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/tomas-rosicky/...,GB1,Arsenal Football Club,350000.0,17500000.0
8,162,Marc,Ziegler,Marc Ziegler,2012,79,marc-ziegler,Germany,Blieskastel,Germany,...,right,193.0,NaN,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/marc-ziegler/p...,L1,Verein für Bewegungsspiele Stuttgart 1893,200000.0,1250000.0
9,215,Roque,Santa Cruz,Roque Santa Cruz,2015,1084,roque-santa-cruz,Paraguay,Asunción,Paraguay,...,right,193.0,2023-12-31 00:00:00,NaN,https://img.a.transfermarkt.technology/portrai...,https://www.transfermarkt.co.uk/roque-santa-cr...,ES1,Málaga CF,250000.0,12000000.0


In [4]:
df_2.head(10)

,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,3333,2004-04-10,7500000,Newcastle United,399,GB1
5,12589,2004-04-10,750000,Heart of Midlothian FC,43,SC1
6,15452,2004-04-10,800000,Getafe CF,6182,NaN
7,15570,2004-04-10,1500000,CSKA Moscow,2410,RU1
8,16136,2004-04-10,250000,LOSC Lille,1082,FR1
9,16756,2004-04-10,50000,Spartak Moskow II,6189,NaN


In [5]:
# Validate merge key and inspect ID coverage
join_key = 'player_id'

if join_key not in df_1.columns or join_key not in df_2.columns:
    raise KeyError(
        f"'{join_key}' must exist in both dataframes. "
        f"df_1 columns: {df_1.columns.tolist()} | df_2 columns: {df_2.columns.tolist()}"
    )

print(f"df_1 shape: {df_1.shape}")
print(f"df_2 shape: {df_2.shape}")
print(f"df_1 unique {join_key}: {df_1[join_key].nunique()}")
print(f"df_2 unique {join_key}: {df_2[join_key].nunique()}")
print(f"df_2 duplicate {join_key} rows: {df_2.duplicated(subset=[join_key]).sum()}")

df_1 shape: (34301, 23)
df_2 shape: (448187, 6)
df_1 unique player_id: 34301
df_2 unique player_id: 31395
df_2 duplicate player_id rows: 416792


In [6]:
# Clean IDs, reduce valuation duplicates to one row per player_id, then merge
df_1_merge = df_1.copy()
df_2_merge = df_2.copy()

for frame in (df_1_merge, df_2_merge):
    frame[join_key] = pd.to_numeric(frame[join_key], errors='coerce').astype('Int64')

df_1_merge = df_1_merge.dropna(subset=[join_key]).copy()
df_2_merge = df_2_merge.dropna(subset=[join_key]).copy()

if 'date' in df_2_merge.columns:
    df_2_merge['date'] = pd.to_datetime(df_2_merge['date'], errors='coerce')
    df_2_latest = (
        df_2_merge
        .sort_values([join_key, 'date'])
        .drop_duplicates(subset=[join_key], keep='last')
    )
else:
    df_2_latest = df_2_merge.drop_duplicates(subset=[join_key], keep='last')

df_merged = df_1_merge.merge(
    df_2_latest,
    on=join_key,
    how='left',
    suffixes=('_player', '_valuation')
)

print(f"Merged shape: {df_merged.shape}")
print(f"Rows with valuation match: {df_merged.filter(like='_valuation').notna().any(axis=1).sum()}")
print(f"Rows without valuation match: {df_merged.filter(like='_valuation').notna().any(axis=1).eq(False).sum()}")

Merged shape: (34301, 28)
Rows with valuation match: 31395
Rows without valuation match: 2906


In [7]:
# Preview and export merged dataset

# Preview and export merged dataset
df_merged.head(10)

,player_id,first_name,last_name,name,last_season,current_club_id_player,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,url,current_club_domestic_competition_id,current_club_name_player,market_value_in_eur_player,highest_market_value_in_eur,date,market_value_in_eur_valuation,current_club_name_valuation,current_club_id_valuation,player_club_domestic_competition_id
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0,2016-01-04,1000000.0,SS Lazio,398.0,IT1
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0,2017-12-28,750000.0,Borussia Dortmund,16.0,L1
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0,2016-06-21,1000000.0,PAOK Thessaloniki,1091.0,GR1
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0,2016-11-15,200000.0,FC Goa,506.0,IT1
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0,2018-06-05,100000.0,Bayern Munich,27.0,L1
5,109,NaN,Dedê,Dedê,2013,825,dede,Brazil,Belo Horizonte,Brazil,...,https://www.transfermarkt.co.uk/dede/profil/sp...,TR1,Eskisehirspor,400000.0,9500000.0,2014-01-27,400000.0,Eskisehirspor,825.0,TR1
6,123,Christoph,Metzelder,Christoph Metzelder,2012,33,christoph-metzelder,Germany,Haltern,Germany,...,https://www.transfermarkt.co.uk/christoph-metz...,L1,FC Schalke 04,1500000.0,9500000.0,2013-01-13,1500000.0,FC Schalke 04,33.0,L1
7,132,Tomas,Rosicky,Tomas Rosicky,2015,11,tomas-rosicky,CSSR,Praha,Czech Republic,...,https://www.transfermarkt.co.uk/tomas-rosicky/...,GB1,Arsenal Football Club,350000.0,17500000.0,2017-06-10,350000.0,AC Sparta Prague,11.0,GB1
8,162,Marc,Ziegler,Marc Ziegler,2012,79,marc-ziegler,Germany,Blieskastel,Germany,...,https://www.transfermarkt.co.uk/marc-ziegler/p...,L1,Verein für Bewegungsspiele Stuttgart 1893,200000.0,1250000.0,2013-01-13,200000.0,VfB Stuttgart,79.0,L1
9,215,Roque,Santa Cruz,Roque Santa Cruz,2015,1084,roque-santa-cruz,Paraguay,Asunción,Paraguay,...,https://www.transfermarkt.co.uk/roque-santa-cr...,ES1,Málaga CF,250000.0,12000000.0,2023-06-30,250000.0,Club Libertad Asunción,1084.0,ES1


In [8]:
cols_to_drop = ['first_name', 'last_name', 'current_club_id_player', 'player_code', 'country_of_birth', 'city_of_birth', 'position', 'foot', 'agent_name', 'image_url', 'url', 'current_club_domestic_competition_id', 'market_value_in_eur_player', 'highest_market_value_in_eur', 'current_club_id_valuation', 'player_club_domestic_competition_id', 'current_club_name_player', 'height_in_cm']

df_merged = df_merged.drop(columns=cols_to_drop, errors='ignore')
df_merged.head(25)

,player_id,name,last_season,country_of_citizenship,date_of_birth,sub_position,contract_expiration_date,date,market_value_in_eur_valuation,current_club_name_valuation
0,10,Miroslav Klose,2015,Germany,1978-06-09 00:00:00,Centre-Forward,NaN,2016-01-04,1000000.0,SS Lazio
1,26,Roman Weidenfeller,2017,Germany,1980-08-06 00:00:00,Goalkeeper,NaN,2017-12-28,750000.0,Borussia Dortmund
2,65,Dimitar Berbatov,2015,Bulgaria,1981-01-30 00:00:00,Centre-Forward,NaN,2016-06-21,1000000.0,PAOK Thessaloniki
3,77,Lúcio,2012,Brazil,1978-05-08 00:00:00,Centre-Back,NaN,2016-11-15,200000.0,FC Goa
4,80,Tom Starke,2017,Germany,1981-03-18 00:00:00,Goalkeeper,NaN,2018-06-05,100000.0,Bayern Munich
5,109,Dedê,2013,Brazil,1978-04-18 00:00:00,Left-Back,NaN,2014-01-27,400000.0,Eskisehirspor
6,123,Christoph Metzelder,2012,Germany,1980-11-05 00:00:00,Centre-Back,NaN,2013-01-13,1500000.0,FC Schalke 04
7,132,Tomas Rosicky,2015,Czech Republic,1980-10-04 00:00:00,Attacking Midfield,NaN,2017-06-10,350000.0,AC Sparta Prague
8,162,Marc Ziegler,2012,Germany,1976-06-13 00:00:00,Goalkeeper,NaN,2013-01-13,200000.0,VfB Stuttgart
9,215,Roque Santa Cruz,2015,Paraguay,1981-08-16 00:00:00,Centre-Forward,2023-12-31 00:00:00,2023-06-30,250000.0,Club Libertad Asunción


In [9]:
#Drop all entries with a missing contract_expiration_date, date_of_birth and/or 'date
df_merged = df_merged.dropna(subset=['contract_expiration_date', 'date_of_birth', 'date', 'sub_position', 'country_of_citizenship'])
print(f"Shape after dropping rows with missing contract_expiration_date, date_of_birth, or date: {df_merged.shape}")

Shape after dropping rows with missing contract_expiration_date, date_of_birth, or date: (19638, 10)


In [10]:
#Convert Date Columns
df_merged['date'] = pd.to_datetime(df_merged['date'])
df_merged['date_of_birth'] = pd.to_datetime(df_merged['date_of_birth'])
df_merged['contract_expiration_date'] = pd.to_datetime(df_merged['contract_expiration_date'])

In [11]:
#Create Age at Valuation and Contract years left using today's date
#df_merged['age_at_valuation'] = (df_merged['date'] - df_merged['date_of_birth']).dt.days / 365.25
#df_merged['contract_years_left'] = (df_merged['contract_expiration_date'] - df_merged['date']).dt.days / 365.25

#Drop rows with contract_years_left more than 10 years
#df_merged = df_merged[df_merged['contract_years_left'] <= 10]

# ...existing code...

#Create Age and Contract years left columns using today's date
today = pd.Timestamp.today().normalize()

df_merged['age'] = (today - df_merged['date_of_birth']).dt.days / 365.25
df_merged['contract_years_left'] = (df_merged['contract_expiration_date'] - today).dt.days / 365.25

#Get age at the time of valuation
df_merged['age_at_valuation'] = (df_merged['date'] - df_merged['date_of_birth']).dt.days / 365.25

#Drop rows with contract_years_left more than 10 years
df_merged = df_merged[df_merged['contract_years_left'] <= 10]

# ...existing code...

In [12]:
df_merged.head(10)

,player_id,name,last_season,country_of_citizenship,date_of_birth,sub_position,contract_expiration_date,date,market_value_in_eur_valuation,current_club_name_valuation,age,contract_years_left,age_at_valuation
9,215,Roque Santa Cruz,2015,Paraguay,1981-08-16,Centre-Forward,2023-12-31,2023-06-30,250000.0,Club Libertad Asunción,44.542094,-2.168378,41.869952
98,2138,Mladen Kascelan,2017,Montenegro,1983-02-13,Defensive Midfield,2023-06-30,2022-12-08,25000.0,Baltika Kaliningrad,43.047228,-2.672142,39.816564
122,2857,Eldin Jakupovic,2022,Switzerland,1984-10-02,Goalkeeper,2023-12-31,2023-08-22,100000.0,Los Angeles FC,41.412731,-2.168378,38.885695
150,3159,Valerio Di Cesare,2018,Italy,1983-05-23,Centre-Back,2024-06-30,2023-06-12,200000.0,SSC Bari,42.776181,-1.670089,40.054757
188,3333,James Milner,2025,England,1986-01-04,Central Midfield,2026-06-30,2025-09-12,750000.0,Brighton & Hove Albion,40.156057,0.328542,39.687885
251,3755,Óscar Ustari,2013,Argentina,1986-07-03,Goalkeeper,2023-12-31,2023-10-25,600000.0,Without Club,39.663244,-2.168378,37.311431
292,4042,Brad Jones,2017,Australia,1982-03-19,Goalkeeper,2023-06-30,2023-04-05,100000.0,Perth Glory,43.953457,-2.672142,41.045859
400,5017,Niklas Moisander,2020,Finland,1985-09-29,Centre-Back,2023-12-31,2023-06-27,300000.0,Malmö FF,40.421629,-2.168378,37.741273
401,5023,Gianluigi Buffon,2020,Italy,1978-01-28,Goalkeeper,2024-06-30,2023-06-12,1000000.0,Parma Calcio 1913,48.090349,-1.670089,45.368925
417,5336,Anastasios Tsokanis,2025,Greece,1991-05-02,Defensive Midfield,2027-06-30,2025-12-12,200000.0,Volos NPS,34.833676,1.327858,34.614648


In [14]:
df_merged['contract_years_left'] = df_merged['contract_years_left'].clip(lower=0)
df_merged['is_free_agent'] = (df_merged['contract_years_left'] == 0).astype(int)
df_merged = df_merged[df_merged['contract_expiration_date'] >= df_merged['date_of_birth']]

#Other derived features
#Add age squared to capture potential non-linear effects of age on market value
df_merged["age_sq"] = df_merged["age_at_valuation"] ** 2

#Age bands to capture non-linear age effects and potential career stage differences
df_merged["age_band"] = pd.cut(
    df_merged["age_at_valuation"],
    bins=[0, 20, 24, 28, 32, 36, 100],
    labels=["teen","young","prime","late_prime","veteran","elder"]
)

#Years from age 27, which is often considered the peak age for footballers, to capture how far players are from their peak
df_merged["years_from_27"] = df_merged["age_at_valuation"] - 27
df_merged["abs_years_from_27"] = df_merged["years_from_27"].abs()

#Short contract indicator for players with less than 1 year left on their contract, which may decrease transfer value due to imminent free agency
df_merged["short_contract"] = (df_merged["contract_years_left"] < 1).astype(int)

#Long contract indicator for players with 4 or more years left on their contract, which may increase transfer value due to long-term security for the selling club
df_merged["long_contract"] = (df_merged["contract_years_left"] >= 4).astype(int)

#Square root of contract years left to capture diminishing returns of additional contract length on transfer value
df_merged["contract_sqrt"] = np.sqrt(df_merged["contract_years_left"])

#Valuation year to capture temporal trends in transfer values and market conditions
df_merged["valuation_year"] = df_merged["date"].dt.year

#Years since 2012 to capture how transfer market values have evolved over time, with 2012 as a reference point for recent market conditions
df_merged["years_since_2012"] = df_merged["valuation_year"] - 2012

In [15]:
df_merged.head(10)

,player_id,name,last_season,country_of_citizenship,date_of_birth,sub_position,contract_expiration_date,date,market_value_in_eur_valuation,current_club_name_valuation,...,is_free_agent,age_sq,age_band,years_from_27,abs_years_from_27,short_contract,long_contract,contract_sqrt,valuation_year,years_since_2012
9,215,Roque Santa Cruz,2015,Paraguay,1981-08-16,Centre-Forward,2023-12-31,2023-06-30,250000.0,Club Libertad Asunción,...,1,1753.092888,elder,14.869952,14.869952,1,0,0.000000,2023,11
98,2138,Mladen Kascelan,2017,Montenegro,1983-02-13,Defensive Midfield,2023-06-30,2022-12-08,25000.0,Baltika Kaliningrad,...,1,1585.358769,elder,12.816564,12.816564,1,0,0.000000,2022,10
122,2857,Eldin Jakupovic,2022,Switzerland,1984-10-02,Goalkeeper,2023-12-31,2023-08-22,100000.0,Los Angeles FC,...,1,1512.097255,elder,11.885695,11.885695,1,0,0.000000,2023,11
150,3159,Valerio Di Cesare,2018,Italy,1983-05-23,Centre-Back,2024-06-30,2023-06-12,200000.0,SSC Bari,...,1,1604.383560,elder,13.054757,13.054757,1,0,0.000000,2023,11
188,3333,James Milner,2025,England,1986-01-04,Central Midfield,2026-06-30,2025-09-12,750000.0,Brighton & Hove Albion,...,0,1575.128217,elder,12.687885,12.687885,1,0,0.573186,2025,13
251,3755,Óscar Ustari,2013,Argentina,1986-07-03,Goalkeeper,2023-12-31,2023-10-25,600000.0,Without Club,...,1,1392.142848,elder,10.311431,10.311431,1,0,0.000000,2023,11
292,4042,Brad Jones,2017,Australia,1982-03-19,Goalkeeper,2023-06-30,2023-04-05,100000.0,Perth Glory,...,1,1684.762541,elder,14.045859,14.045859,1,0,0.000000,2023,11
400,5017,Niklas Moisander,2020,Finland,1985-09-29,Centre-Back,2023-12-31,2023-06-27,300000.0,Malmö FF,...,1,1424.403695,elder,10.741273,10.741273,1,0,0.000000,2023,11
401,5023,Gianluigi Buffon,2020,Italy,1978-01-28,Goalkeeper,2024-06-30,2023-06-12,1000000.0,Parma Calcio 1913,...,1,2058.339391,elder,18.368925,18.368925,1,0,0.000000,2023,11
417,5336,Anastasios Tsokanis,2025,Greece,1991-05-02,Defensive Midfield,2027-06-30,2025-12-12,200000.0,Volos NPS,...,0,1198.173822,veteran,7.614648,7.614648,0,0,1.152327,2025,13


In [16]:
output_path = '../transfermarkt_merged_players_with_valuation.csv'
df_merged.to_csv(output_path, index=False)
print(f"Saved merged dataset to: {output_path}")

Saved merged dataset to: ../transfermarkt_merged_players_with_valuation.csv
